# Lesson 19 - SQLite and Persistent Records

**Goal:** Store structured data with constraints, parameterized SQL, and explicit transaction handling.

**Level:** intermediate. Complete the preceding chapters first. This notebook defines its own
inputs and runs independently. Run cells top to bottom with **Shift+Enter**. Predict results,
inspect the output, and change one input at a time. No external services or third-party packages
are required beyond your Jupyter setup.

Examples include lightweight executable checks. Practice cells intentionally contain no solutions;
expected results help you verify your own work. Temporary files and databases are isolated and cleaned up.

Where salary examples appear, use the course's simplified exercise rules: DA 48% of basic,
HRA 16% capped at 16,000 monthly, PF 12% of basic plus DA, and annual tax 20% only on annual
gross above 12,00,000, divided by 12 for monthly tax. PF is a separate deduction.
These are learning assumptions, not actual tax or payroll rules.

## 1. Create an isolated database

sqlite3 is included with Python. An in-memory database keeps this notebook repeatable. Store money as integer cents here to avoid binary-float storage ambiguity. Define primary keys and checks so the database also enforces important rules.

In [1]:
import sqlite3
from contextlib import closing

with closing(sqlite3.connect(":memory:")) as connection:
    connection.execute("""CREATE TABLE employees (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        department TEXT NOT NULL,
        basic_cents INTEGER NOT NULL CHECK(basic_cents >= 0)
    )""")
    with connection:
        connection.execute("INSERT INTO employees VALUES (?, ?, ?, ?)",
                           (101, "Asha", "Finance", 5_250_000))
    print(connection.execute("SELECT id, name, basic_cents FROM employees").fetchall())

[(101, 'Asha', 5250000)]


## 2. Parameters, filtering, and ordering

Bind values with placeholders rather than formatting them into SQL. Parameters represent values, not table or column names. ORDER BY is needed for guaranteed result ordering. A connection context handles commit or rollback, but does not close the connection; closing(...) does.

In [2]:
rows = [(101, "Asha", "Finance", 5_250_000),
        (102, "Arun", "IT", 7_500_000),
        (103, "Neha", "Finance", 6_000_000)]
with closing(sqlite3.connect(":memory:")) as connection:
    connection.execute("CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, department TEXT, basic_cents INTEGER CHECK(basic_cents >= 0))")
    with connection:
        connection.executemany("INSERT INTO employees VALUES (?, ?, ?, ?)", rows)
    selected = connection.execute(
        "SELECT name, basic_cents FROM employees WHERE department = ? ORDER BY id",
        ("Finance",)).fetchall()
    print(selected)
assert selected == [("Asha", 5_250_000), ("Neha", 6_000_000)]

[('Asha', 5250000), ('Neha', 6000000)]


## 3. Transactions and rollback

A transaction groups changes. If an integrity error escapes a connection context, those changes roll back. Catch the error outside the with block so the context sees it. Avoid relying on partially inserted data after a failed transaction.

In [3]:
with closing(sqlite3.connect(":memory:")) as connection:
    connection.execute("CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT)")
    try:
        with connection:
            connection.execute("INSERT INTO employees VALUES (?, ?)", (101, "Asha"))
            connection.execute("INSERT INTO employees VALUES (?, ?)", (101, "Duplicate"))
    except sqlite3.IntegrityError:
        print("Duplicate ID: transaction rolled back.")
    count = connection.execute("SELECT COUNT(*) FROM employees").fetchone()[0]
    print("Rows remaining:", count)
assert count == 0

Duplicate ID: transaction rolled back.
Rows remaining: 0


## 4. Aggregate in SQL

GROUP BY calculates per-group summaries. SUM on no matching rows is NULL; COALESCE can define a zero result. Keep cents as integers until formatting.

In [4]:
with closing(sqlite3.connect(":memory:")) as connection:
    connection.execute("CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, department TEXT, basic_cents INTEGER)")
    with connection:
        connection.executemany("INSERT INTO employees VALUES (?, ?, ?, ?)", rows)
    totals = connection.execute("SELECT department, COUNT(*), SUM(basic_cents) FROM employees GROUP BY department ORDER BY department").fetchall()
    empty_total = connection.execute("SELECT COALESCE(SUM(basic_cents), 0) FROM employees WHERE department = ?", ("Legal",)).fetchone()[0]
    print(totals)
    print("Empty total:", empty_total)
assert empty_total == 0

[('Finance', 2, 11250000), ('IT', 1, 7500000)]
Empty total: 0


## Practice lab

Define all inputs in each answer cell, use `practice_` names where helpful, and explain important
decisions in comments. Try a task before looking at its expected results. Add at least one normal,
one boundary, and one invalid-input check where the task has an input contract.
Keep file exercises inside TemporaryDirectory until you deliberately choose a permanent output path.

### Exercise 1. Parameterized lookup

Query an employee by ID using a placeholder and handle no matching row.

**Check:** Existing ID returns one record; missing ID returns None from fetchone.

In [5]:
# Write your solution and checks here.

### Exercise 2. Update pay

Update one salary in cents in a transaction and query it afterward.

**Check:** Basic 52,500.00 becomes 55,000.00 as 5,500,000 cents.

In [6]:
# Write your solution and checks here.

### Exercise 3. Constraint failure

Attempt a duplicate ID and a negative salary under constraints.

**Check:** Both raise IntegrityError and do not leave partial transaction changes.

In [7]:
# Write your solution and checks here.

### Exercise 4. Group totals

Calculate count and total basic cents per department.

**Check:** Finance: two employees, 11,250,000 cents in the sample.

In [8]:
# Write your solution and checks here.

### Exercise 5. File-backed round trip

Use a database file inside TemporaryDirectory, close it, reopen it, and query stored records.

**Check:** Records survive reconnection within the directory lifetime; nothing remains after cleanup.

In [9]:
# Write your solution and checks here.

## Chapter project

Import validated CSV records into a temporary SQLite database, reject duplicate IDs, and export an ordered department summary. Use integer cents and document whether invalid rows reject the whole batch or are skipped.

In [10]:
# Build your chapter project here. Separate inputs, logic, and display.

## Reflection

1. Which input assumptions does your solution enforce, and which does it document?
2. What happens for empty input, a boundary value, or a malformed record?
3. Which parts can be reused without the notebook?
4. What would make the code clearer to another learner?

Restart the kernel and run all cells after completing your work to catch hidden dependencies.

**Next:** `20_payroll_capstone.ipynb` - Capstone: A Validated Payroll Report.